In [2]:
import polars as pl

caminho = r"D:\ADMStoIQS\data\mart\apuracao\agrupamento_oms_APURACAO_202605.parquet"
formato_data = "%Y-%m-%d %H:%M:%S"

# Vamos trazer apenas uma amostra para checar a saúde dos dados
df_checagem = (
    pl.scan_parquet(caminho)
    .select(["NUM_OCORRENCIA_ADMS", "DATA_HORA_INIC_INTRP", "DATA_HORA_FIM_INTRP"])
    .head(5) # Pega as 5 primeiras linhas
    .with_columns([
        pl.col("DATA_HORA_INIC_INTRP").str.to_datetime(formato_data, strict=False).alias("inicio_convertido"),
        pl.col("DATA_HORA_FIM_INTRP").str.to_datetime(formato_data, strict=False).alias("fim_convertido")
    ])
    .collect()
)

print(df_checagem)

shape: (5, 5)
┌────────────────────┬────────────────────┬───────────────────┬───────────────────┬────────────────┐
│ NUM_OCORRENCIA_ADM ┆ DATA_HORA_INIC_INT ┆ DATA_HORA_FIM_INT ┆ inicio_convertido ┆ fim_convertido │
│ S                  ┆ RP                 ┆ RP                ┆ ---               ┆ ---            │
│ ---                ┆ ---                ┆ ---               ┆ datetime[μs]      ┆ datetime[μs]   │
│ str                ┆ str                ┆ str               ┆                   ┆                │
╞════════════════════╪════════════════════╪═══════════════════╪═══════════════════╪════════════════╡
│ 1-26285845         ┆ 12/05/2026         ┆ 12/05/2026        ┆ null              ┆ null           │
│                    ┆ 18:25:14           ┆ 18:25:24          ┆                   ┆                │
│ 1-26285845         ┆ 12/05/2026         ┆ 12/05/2026        ┆ null              ┆ null           │
│                    ┆ 18:25:14           ┆ 18:25:24          ┆              

In [3]:
import polars as pl

caminho = r"D:\ADMStoIQS\data\mart\apuracao\agrupamento_oms_APURACAO_202605.parquet"
formato_data = "%Y-%m-%d %H:%M:%S"

# Filtrando os registros com inconsistência de data (duração negativa)
df_negativos = (
    pl.scan_parquet(caminho)
    .select(["NUM_OCORRENCIA_ADMS", "DATA_HORA_INIC_INTRP", "DATA_HORA_FIM_INTRP"])
    .with_columns([
        pl.col("DATA_HORA_INIC_INTRP").str.to_datetime(formato_data, strict=False).alias("inicio_convertido"),
        pl.col("DATA_HORA_FIM_INTRP").str.to_datetime(formato_data, strict=False).alias("fim_convertido")
    ])
    # Filtra onde o fim ocorre antes do início
    .filter(pl.col("fim_convertido") < pl.col("inicio_convertido"))
    .collect()
)

# Verifica a quantidade de registros encontrados
print(f"Quantidade de registros com duração negativa: {len(df_negativos)}")

# Exibe os registros encontrados
print(df_negativos)

Quantidade de registros com duração negativa: 0
shape: (0, 5)
┌────────────────────┬────────────────────┬───────────────────┬───────────────────┬────────────────┐
│ NUM_OCORRENCIA_ADM ┆ DATA_HORA_INIC_INT ┆ DATA_HORA_FIM_INT ┆ inicio_convertido ┆ fim_convertido │
│ S                  ┆ RP                 ┆ RP                ┆ ---               ┆ ---            │
│ ---                ┆ ---                ┆ ---               ┆ datetime[μs]      ┆ datetime[μs]   │
│ str                ┆ str                ┆ str               ┆                   ┆                │
╞════════════════════╪════════════════════╪═══════════════════╪═══════════════════╪════════════════╡
└────────────────────┴────────────────────┴───────────────────┴───────────────────┴────────────────┘


In [4]:
df_negativos = (
    pl.scan_parquet(caminho)
    .select(["NUM_SEQ_INTRP", "NUM_OCORRENCIA_ADMS", "DATA_HORA_INIC_INTRP", "DATA_HORA_FIM_INTRP"])
    .with_columns([
        pl.coalesce([
            pl.col("DATA_HORA_INIC_INTRP").str.to_datetime("%Y-%m-%d %H:%M:%S", strict=False),
            pl.col("DATA_HORA_INIC_INTRP").str.to_datetime("%d/%m/%Y %H:%M:%S", strict=False),
        ]).alias("inicio_convertido"),
        pl.coalesce([
            pl.col("DATA_HORA_FIM_INTRP").str.to_datetime("%Y-%m-%d %H:%M:%S", strict=False),
            pl.col("DATA_HORA_FIM_INTRP").str.to_datetime("%d/%m/%Y %H:%M:%S", strict=False),
        ]).alias("fim_convertido"),
    ])
    .with_columns(
        (pl.col("fim_convertido") - pl.col("inicio_convertido"))
        .dt.total_minutes()
        .alias("duracao_minutos")
    )
    .filter(pl.col("duracao_minutos") < 0)
    .collect()
)

print(f"Registros negativos: {len(df_negativos)}")
print(f"NUM_SEQ_INTRP distintas: {df_negativos.select('NUM_SEQ_INTRP').unique().height}")

Registros negativos: 3963
NUM_SEQ_INTRP distintas: 3
